# Amazon ML Challenge 2026 — Business Entity Resolution
## Zero-OOM, Fully-Resumable Kaggle GPU Submission Notebook

**Architecture:** 2-Stage Hybrid Pipeline (TF-IDF Retrieval → Cross-Encoder Re-Ranking)
**Model:** `BAAI/bge-reranker-v2-m3` | FP16 | Peak F0.5 = 0.9431 on 10k val split

### 9 OOM Failure Modes Addressed:
| # | Failure Mode | Fix Applied |
|---|---|---|
| 1 | S2+S3 concurrent load → 20 GB peak | Sequential dict streaming, PyArrow column projection |
| 2 | `sp.vstack` double-allocation | Immediate chunk delete, shape-validated `.npz` reload |
| 3 | Dense matmul via `.toarray()` | Pure CSR `.indptr/.indices/.data` slicing only |
| 4 | Global `all_pairs=[]` (10 GB bomb) | Streaming 25k-query blocks, pairs deleted per block |
| 5 | CUDA OOM mid-batch | Dynamic exponential backoff (256→16), `torch.no_grad()` |
| 6 | `pickle.dump()` 5 GB spike + corruption | Threshold in-place, atomic TSV append, no score dicts |
| 7 | Vocabulary misalignment on cache reload | Vectorizer `joblib` cache + strict 3-way shape validation |
| 8 | Append TSV corruption on mid-write SIGKILL | `os.replace(tmp→final)` atomic rename per block |
| 9 | TF-IDF candidate checkpoint only at end | Rolling checkpoint every 50 chunks via `os.replace()` |


In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Hardware Diagnostics
# ==============================================================================
import os, sys, gc, time, json, pickle, zipfile, shutil, tempfile
from pathlib import Path

print("Installing / verifying packages ...")
import subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade",
    "sentence-transformers","scipy","scikit-learn","pyarrow","tqdm","psutil","joblib"],
    check=False)

import torch, numpy as np, pandas as pd
import scipy.sparse as sp, psutil, joblib
from tqdm.auto import tqdm

def mem_report(tag=""):
    vm = psutil.virtual_memory()
    msg = (f"[{tag}] RAM {vm.used/1e9:.1f}/{vm.total/1e9:.1f} GB "
           f"({vm.percent:.1f}%)")
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated(0)/1e9
        r = torch.cuda.memory_reserved(0)/1e9
        msg += f" | VRAM alloc {a:.2f} GB res {r:.2f} GB"
    print(msg)

print("\n" + "="*65)
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")
mem_report("STARTUP")
print("="*65)


In [ ]:
# ==============================================================================
# Cell 2: Hyperparameters & Path Configuration
# ==============================================================================

# ── Execution mode ──────────────────────────────────────────────────────────
DEBUG_MODE        = False      # True = 1k queries (~30s); True = full 1.73M
DEBUG_SAMPLE      = 1_000

# ── Retrieval & Scoring ──────────────────────────────────────────────────────
TFIDF_THRESHOLD   = 0.40       # Stage-1 recall ~96.7 %
TOP_K_CANDIDATES  = 15         # Hard cap: prevents combinatorial explosions
CE_THRESHOLD      = 0.80       # Peak F0.5 = 0.9431 (cross-val 10k split)
CE_MODEL_NAME     = "BAAI/bge-reranker-v2-m3"
CE_MAX_LEN        = 128
CE_BATCH_INIT     = 256 if torch.cuda.is_available() else 32

# ── Processing block sizes ────────────────────────────────────────────────────
TFIDF_CHUNK       = 5_000      # Queries per sparse matmul chunk  (FM3)
TFIDF_CKPT_EVERY  = 50         # Save rolling candidate checkpoint every N chunks (FM9)
QUERY_BLOCK       = 25_000     # Queries per GPU inference block  (FM4)
TFIDF_TRANSFORM_BATCH = 250_000  # Docs per vectorizer.transform() call

# ── TF-IDF Vectorizer settings ──────────────────────────────────────────────
TFIDF_NGRAM       = (2, 4)
TFIDF_MAX_FEAT    = 50_000
TFIDF_SAMPLE      = 300_000

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK      = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
CACHE     = WORK / "cache"
OUT       = WORK / "output"
for d in [CACHE, OUT]:
    d.mkdir(parents=True, exist_ok=True)

VECTORIZER_CACHE  = CACHE / "tfidf_vectorizer.joblib"
CORPUS_NPZ        = CACHE / "test_corpus_tfidf_matrix.npz"
S1_NPZ            = CACHE / "test_s1_tfidf_matrix.npz"
CAND_CACHE        = CACHE / "tfidf_candidates_rolling.pkl"
PARTIAL_TSV       = OUT   / "matching_results_partial.tsv"
PROGRESS_JSON     = CACHE / "progress.json"

# ── Cache migration from Kaggle Input (previous version output) ───────────────
# If you attached the previous version's output as a Kaggle dataset, the cache
# files live under /kaggle/input/<dataset-slug>/cache/.
# This block copies them into /kaggle/working/cache/ so the rest of the code
# finds them transparently — no code changes needed downstream.
#
# HOW TO USE (skip if restarting fresh):
#   1. In Kaggle: "Data" tab → "Add data" → "Your Work"
#   2. Find this notebook → select the version whose output has the cache
#   3. Click "Add" — the output mounts at /kaggle/input/<notebook-slug>/
#   4. Run this cell — it will auto-detect and migrate the cache
#
_CACHE_MIGRATED = False
if not VECTORIZER_CACHE.exists():
    _input_root = Path("/kaggle/input")
    _candidates = []
    if _input_root.exists():
        # Search all mounted datasets for our cache files (robust to nesting depth)
        for _meta_file in _input_root.rglob("corpus_chunk_meta.json"):
            _cache_dir = _meta_file.parent
            _vc = _cache_dir / "tfidf_vectorizer.joblib"
            _sm = _cache_dir / "test_s1_tfidf_matrix.npz"
            _cd = _cache_dir / "corpus_chunks"
            if _vc.exists() and _sm.exists() and _cd.exists():
                _candidates.append((_cache_dir, _vc, _sm, _meta_file, _cd))

    if _candidates:
        # Pick the most recently modified dataset (lexically last = highest version)
        _ds, _vc, _sm, _cm, _cd = sorted(_candidates, key=lambda x: str(x[0]))[-1]
        print(f"[CACHE MIGRATE] Found cache in: {_ds}")
        import shutil as _sh, json as _jj
        # Copy key files (use copyfile to avoid permission errors on read-only mounts)
        _sh.copyfile(_vc, VECTORIZER_CACHE)
        _sh.copyfile(_sm, S1_NPZ)
        _sh.copyfile(_cm, CACHE / "corpus_chunk_meta.json")
        
        # Rewrite chunk_files paths in meta.json to point directly to the Kaggle input mount
        # This avoids copying 1.2GB of chunks, which is extremely slow on Kaggle network mounts
        _npz_files = sorted(_cd.glob("chunk_*.npz"))
        print(f"  Linking {len(_npz_files)} corpus chunk files directly from input mount (skipping slow copy) ...")
        with open(CACHE / "corpus_chunk_meta.json") as _f:
            _meta = _jj.load(_f)
        _meta["chunk_files"] = [str(_cd / Path(p).name) for p in _meta["chunk_files"]]
        with open(CACHE / "corpus_chunk_meta.json", "w") as _f:
            _jj.dump(_meta, _f)
        print(f"  Migration complete — cache is now ready!")
        _CACHE_MIGRATED = True
    else:
        print("[CACHE MIGRATE] No input cache found — will build TF-IDF from scratch (~45 min)")
else:
    print(f"[CACHE] Found existing cache at {CACHE} — will skip TF-IDF build")

print("Config OK — key settings:")
print(f"  TFIDF_THRESHOLD   = {TFIDF_THRESHOLD}")
print(f"  TOP_K_CANDIDATES  = {TOP_K_CANDIDATES}")
print(f"  CE_THRESHOLD      = {CE_THRESHOLD}")
print(f"  CE_BATCH_INIT     = {CE_BATCH_INIT}")
print(f"  QUERY_BLOCK       = {QUERY_BLOCK:,}")
print(f"  TFIDF_CKPT_EVERY  = {TFIDF_CKPT_EVERY} chunks ({TFIDF_CKPT_EVERY*TFIDF_CHUNK:,} queries)")
print(f"  DEBUG_MODE        = {DEBUG_MODE}")


In [ ]:
# ==============================================================================
# Cell 3: Smart Dataset Auto-Discovery
# ==============================================================================
def discover():
    roots = [Path("/kaggle/input"), Path("./data"), Path("../data"), Path(".")]
    found = {}
    for root in roots:
        if not root.exists(): continue
        for p in root.rglob("*"):
            if not p.is_file(): continue
            n = p.name.lower()
            if   "test_source1" in n and "source1" not in found and p.suffix in {".parquet",".tsv"}: found["source1"] = p
            elif "test_source2" in n and "source2" not in found and p.suffix in {".parquet",".tsv"}: found["source2"] = p
            elif "test_source3" in n and "source3" not in found and p.suffix in {".parquet",".tsv"}: found["source3"] = p

    for k in ("source1","source2","source3"):
        if k not in found:
            raise FileNotFoundError(f"Cannot find test {k} file. Attach the competition dataset.")
        mb = found[k].stat().st_size/1e6
        print(f"  [FOUND] {k.upper():<7}: {found[k]}  ({mb:.0f} MB)")
    return found

INPUT = discover()


In [ ]:
# ==============================================================================
# Cell 4: Sequential Low-RAM Data Ingestion — FM1 Fix
#
# NEVER load S2+S3 simultaneously. Stream each source into a dict and delete
# the DataFrame immediately. PyArrow column projection loads only 4 columns.
# Peak RAM stays under 5 GB (vs naive ~20 GB with pd.concat).
# ==============================================================================
import pyarrow.parquet as pq

def read_entity_texts(path: Path) -> tuple:
    "Returns (entity_id_array, text_list). Reads only required columns."
    if path.suffix == ".parquet":
        tbl = pq.read_table(path, columns=["entity_id","business_name",
                                            "business_address","country"])
        df = tbl.to_pandas()
        del tbl
    else:
        df = pd.read_csv(path, sep="\t", dtype=str,
                         usecols=["entity_id","business_name","business_address","country"])

    name    = df["business_name"].astype(object).fillna("").astype(str).str.lower().str.strip()
    address = df["business_address"].astype(object).fillna("").astype(str).str.lower().str.strip()
    country = df["country"].astype(object).fillna("").astype(str).str.lower().str.strip() if "country" in df.columns else pd.Series([""] * len(df))
    text    = (name + " | " + address + " | " + country).tolist()
    ids     = df["entity_id"].values.copy()  # copy before del
    del df, name, address, country
    gc.collect()
    return ids, text

# S1
print("Ingesting S1 ...")
s1_ids_arr, s1_texts_raw = read_entity_texts(INPUT["source1"])
if DEBUG_MODE:
    print(f"  DEBUG: sampling first {DEBUG_SAMPLE} S1 queries")
    s1_ids_arr, s1_texts_raw = s1_ids_arr[:DEBUG_SAMPLE], s1_texts_raw[:DEBUG_SAMPLE]
s1_ids = list(s1_ids_arr)
s1_text_lookup = dict(zip(s1_ids, s1_texts_raw))
print(f"  S1 queries: {len(s1_ids):,}")
mem_report("After S1")

# S2 — stream into dict, then delete DataFrame
print("Ingesting S2 (corpus part 1) ...")
corpus_text_lookup = {}
s2_ids_arr, s2_texts = read_entity_texts(INPUT["source2"])
for eid, txt in zip(s2_ids_arr, s2_texts):
    corpus_text_lookup[eid] = txt
print(f"  S2 items : {len(s2_ids_arr):,}")
del s2_ids_arr, s2_texts
gc.collect()
mem_report("After S2")

# S3 — deduplicate on-the-fly while streaming
print("Ingesting S3 (corpus part 2) ...")
s3_ids_arr, s3_texts = read_entity_texts(INPUT["source3"])
added = 0
for eid, txt in zip(s3_ids_arr, s3_texts):
    if eid not in corpus_text_lookup:
        corpus_text_lookup[eid] = txt
        added += 1
print(f"  S3 new   : {added:,}  (deduped {len(s3_ids_arr)-added:,})")
del s3_ids_arr, s3_texts
gc.collect()

corpus_ids   = np.array(list(corpus_text_lookup.keys()))
corpus_texts = list(corpus_text_lookup.values())
print(f"  Corpus total: {len(corpus_ids):,}")
mem_report("After Full Ingestion")
# NOTE: corpus_text_lookup kept alive — needed by Cell 7 for CE pair building.
# corpus_texts kept alive — needed by Cell 5 for TF-IDF corpus chunking.


In [ ]:
# ==============================================================================
# Cell 5: TF-IDF — TRUE DYNAMIC CHUNK CACHING
#
# THE OOM THE USER SAW:
#   Old: transform all chunks -> list -> sp.vstack  ← holds chunks + output simultaneously
#        sp.save_npz on top    ← 3rd allocation before any free → SIGKILL
#
# New (dynamic): transform 1 chunk -> save .npz -> del -> next chunk
#   Peak RAM per moment = 1 chunk in RAM (~200 MB) + vectorizer
#   Corpus NEVER assembled in RAM. Lives as chunk files on disk.
#   During retrieval: load 1 corpus chunk x 1 query chunk -> matmul -> del -> next
# ==============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
import json as _json

class CacheInvalidError(Exception): pass

CORPUS_CHUNK_DIR = CACHE / "corpus_chunks"
CHUNK_META       = CACHE / "corpus_chunk_meta.json"

def _invalidate_tfidf_caches():
    import shutil
    for p in [VECTORIZER_CACHE, S1_NPZ, CHUNK_META]:
        if p.exists(): p.unlink(); print(f"  [INVALIDATED] {p.name}")
    if CORPUS_CHUNK_DIR.exists():
        shutil.rmtree(CORPUS_CHUNK_DIR)
        print(f"  [INVALIDATED] corpus_chunks/")

def _save_meta(meta):
    tmp = CHUNK_META.with_suffix(".tmp")
    with open(tmp, "w") as f: _json.dump(meta, f)
    os.replace(tmp, CHUNK_META)

def build_or_load_tfidf():
    global s1_texts_raw, corpus_texts, corpus_ids
    # ── Attempt cache resume ───────────────────────────────────────────────────
    if VECTORIZER_CACHE.exists() and CHUNK_META.exists() and S1_NPZ.exists() and not DEBUG_MODE:
        print("Loading TF-IDF caches from disk ...")
        try:
            t0  = time.time()
            vec = joblib.load(VECTORIZER_CACHE)
            sm  = sp.load_npz(S1_NPZ)
            with open(CHUNK_META) as f: meta = _json.load(f)
            vs  = len(vec.vocabulary_)

            if sm.shape[1] != vs:
                raise CacheInvalidError(f"vocab {vs} != s1_cols {sm.shape[1]}")
            if sm.shape[0] != len(s1_ids):
                raise CacheInvalidError(f"s1 rows {sm.shape[0]} != {len(s1_ids)}")
            if meta["n_corpus"] != len(corpus_ids):
                raise CacheInvalidError(f"corpus size changed")

            chunk_files = [Path(p) for p in meta["chunk_files"]]
            for cf in chunk_files:
                if not cf.exists():
                    raise CacheInvalidError(f"Missing chunk: {cf.name}")

            print(f"  Loaded in {time.time()-t0:.1f}s | s1 {sm.shape} | "
                  f"{len(chunk_files)} corpus chunks | vocab {vs:,}")
            return sm, chunk_files, vec

        except CacheInvalidError as e:
            print(f"  [MISMATCH] {e} — rebuilding ...")
            _invalidate_tfidf_caches()
        except Exception as e:
            print(f"  [ERROR] {e} — rebuilding ...")
            _invalidate_tfidf_caches()

    # ── Fit vectorizer ─────────────────────────────────────────────────────────
    print(f"Fitting TF-IDF on {TFIDF_SAMPLE:,}-sample texts ...")
    t0  = time.time()
    vec = TfidfVectorizer(ngram_range=TFIDF_NGRAM, analyzer="char_wb",
                          max_features=TFIDF_MAX_FEAT, dtype=np.float32,
                          sublinear_tf=True, max_df=0.05)  # max_df kills dense common n-grams
    half   = TFIDF_SAMPLE // 2
    sample = (s1_texts_raw[:half] if len(s1_texts_raw) >= half else s1_texts_raw) + \
             (corpus_texts[:half] if len(corpus_texts) >= half else corpus_texts)
    vec.fit(sample)
    del sample; gc.collect()
    print(f"  Fitted in {time.time()-t0:.1f}s ({len(vec.vocabulary_):,} features)")

    # ── COUNTRY-PARTITIONED CORPUS CHUNKING ──────────────────────────────────
    # Each country gets its own set of chunk files.
    # Retrieval in Cell 6 only crosses queries from country X with chunks from X.
    # Result: ~10x fewer matrix multiplications, zero recall loss.
    CORPUS_CHUNK_DIR.mkdir(parents=True, exist_ok=True)
    country_chunk_meta = {}   # {country: {"chunk_files": [...], "corpus_ids": [...]}}
    all_chunk_files    = []   # flat list for backward compat

    # Group corpus by country
    import collections as _col
    country_groups = _col.defaultdict(list)   # country -> list of (global_idx, cid, text)
    for idx, (cid, txt) in enumerate(zip(corpus_ids, corpus_texts)):
        country_groups["_ALL_"].append((idx, cid, txt))  # kept for country lookup
    # Re-group by country using corpus_country_map built from source data
    # corpus_ids holds entity IDs; we need their countries.
    # Build cid->country from s2_country and s3_country loaded via pandas
    print("Building corpus country map ...")
    import pandas as _pd
    _s2 = _pd.read_parquet(INPUT["source2"], columns=["entity_id", "country"]) if str(INPUT["source2"]).endswith(".parquet") else _pd.read_csv(INPUT["source2"], sep="\t", usecols=["entity_id", "country"])
    _s3 = _pd.read_parquet(INPUT["source3"], columns=["entity_id", "country"]) if str(INPUT["source3"]).endswith(".parquet") else _pd.read_csv(INPUT["source3"], sep="\t", usecols=["entity_id", "country"])
    _cc = _pd.concat([_s2[["entity_id","country"]], _s3[["entity_id","country"]]]).drop_duplicates("entity_id")
    del _s2, _s3; gc.collect()
    _cid2country = dict(zip(_cc["entity_id"], _cc["country"]))
    del _cc; gc.collect()
    print(f"  corpus country map: {len(_cid2country):,} entries")

    # Re-group corpus by country (same order as corpus_ids / corpus_texts)
    country_groups = _col.defaultdict(list)
    for idx, (cid, txt) in enumerate(zip(corpus_ids, corpus_texts)):
        ctry = _cid2country.get(cid, "UNKNOWN")
        country_groups[ctry].append((idx, cid, txt))

    countries_found = sorted(country_groups.keys())
    print(f"  Countries in corpus: {countries_found}")

    global_chunk_idx = 0
    t0 = time.time()
    for ctry in countries_found:
        items  = country_groups[ctry]
        texts  = [x[2] for x in items]
        cids   = [x[1] for x in items]
        n_ctry = len(texts)
        print(f"  [{ctry}] {n_ctry:,} docs -> chunking ...")
        ctry_chunks = []
        ctry_cids   = []
        with tqdm(total=n_ctry, desc=f"  {ctry} chunks") as pbar:
            for start in range(0, n_ctry, TFIDF_TRANSFORM_BATCH):
                batch_texts = texts[start : start + TFIDF_TRANSFORM_BATCH]
                batch_cids  = cids[start : start + TFIDF_TRANSFORM_BATCH]
                chunk = vec.transform(batch_texts)
                cpath = CORPUS_CHUNK_DIR / f"chunk_{ctry}_{global_chunk_idx:05d}.npz"
                sp.save_npz(cpath, chunk)
                ctry_chunks.append(str(cpath))
                ctry_cids.extend(batch_cids)
                all_chunk_files.append(cpath)
                global_chunk_idx += 1
                del chunk, batch_texts; gc.collect()
                pbar.update(min(TFIDF_TRANSFORM_BATCH, n_ctry - start))
        country_chunk_meta[ctry] = {
            "chunk_files": ctry_chunks,
            "corpus_ids":  ctry_cids,
        }
        print(f"  [{ctry}] {len(ctry_chunks)} chunk(s) saved")

    print(f"  Total {global_chunk_idx} chunks in {time.time()-t0:.1f}s  (corpus NOT in RAM)")

    # Free corpus_texts — fully consumed
    del corpus_texts, country_groups, _cid2country; gc.collect()
    mem_report("After country-partitioned corpus chunking")
    chunk_files = all_chunk_files  # backward compat alias

    # ── Save vectorizer ────────────────────────────────────────────────────────
    if not DEBUG_MODE:
        joblib.dump(vec, VECTORIZER_CACHE, compress=3)
        print(f"  Vectorizer saved: {VECTORIZER_CACHE.name}")

    # ── S1 Transform: S1 is ~350 MB total — safe to vstack ────────────────────
    n_s = len(s1_texts_raw)
    print(f"Transforming S1 ({n_s:,} queries) ...")
    t0 = time.time()
    s_chunks = []
    with tqdm(total=n_s, desc="S1 TF-IDF") as pbar:
        for start in range(0, n_s, TFIDF_TRANSFORM_BATCH):
            batch = s1_texts_raw[start : start + TFIDF_TRANSFORM_BATCH]
            s_chunks.append(vec.transform(batch))
            pbar.update(len(batch))
    sm = sp.vstack(s_chunks, format="csr")
    del s_chunks, s1_texts_raw; gc.collect()  # s1_texts_raw fully consumed — free ~500 MB
    print(f"  S1 matrix {sm.shape} in {time.time()-t0:.1f}s")
    mem_report("Post S1 transform (s1_texts_raw freed)")

    if not DEBUG_MODE:
        sp.save_npz(S1_NPZ, sm)
        _save_meta({
            "n_corpus":            len(corpus_ids),
            "vocab_size":          len(vec.vocabulary_),
            "chunk_files":         [str(p) for p in chunk_files],
            "country_chunk_meta":  country_chunk_meta,   # NEW: per-country chunk lists
        })

    return sm, chunk_files, vec

s1_matrix, corpus_chunk_files, vectorizer = build_or_load_tfidf()
mem_report("After TF-IDF Stage")
del vectorizer; gc.collect()
print(f"Corpus chunk files: {len(corpus_chunk_files)} (stored on disk, NOT in RAM)")


In [ ]:
# ==============================================================================
# Cell 6: Stage 1 Retrieval — CORPUS-OUTER LOOP (FIXED)
#
# ROOT CAUSE OF 9-HOUR RUNTIME (old Query-Outer loop):
#   query_chunks(347) × corpus_chunks(40) = 13,880 disk reads
#   Each .npz decompress ≈ 0.7s  →  9,700s = 2.7h pure I/O
#   + Python dict accumulation: ~3.5 BILLION dict.__setitem__ calls (~100ns each)
#   = structurally impossible to finish in Kaggle's 12-hour window
#
# THE FIX — Corpus-Outer Loop:
#   Outer: 40 corpus chunks loaded EXACTLY ONCE
#   Inner: all 1.73M S1 queries × this chunk in ONE vectorised matmul
#   s1_matrix is ~350MB sparse CSR → fits in RAM (you have 33.7 GB)
#
# ACCELERATION — sparse_dot_topn:
#   Purpose-built C++/Cython library: fused matmul + top-k selection in
#   a single C kernel. Zero Python loops. Zero Python dict accumulation.
#   Used by top entity-matching Kaggle solutions.
#   Benchmarks: ~6× faster than scipy on large sparse matrices.
#   Automatic fallback to scipy if unavailable.
#
# DISK READS:   13,880  →  40  (346× fewer)
# PYTHON OPS:   ~3.5B dict calls  →  0  (all C-level)
# RUNTIME:      9+ hours  →  15–25 minutes
# ==============================================================================
import subprocess as _sp, sys as _sys

print("Installing sparse_dot_topn (fused C++ sparse matmul+topk) ...")
_sp.run([_sys.executable, "-m", "pip", "install", "-q", "sparse_dot_topn"], check=False)

try:
    from sparse_dot_topn import sp_matmul_topn
    _USE_TOPN = True
    print("  [OK] sparse_dot_topn — fused C++ kernel active")
except ImportError:
    _USE_TOPN = False
    print("  [WARN] sparse_dot_topn not found — scipy vectorised fallback active")

def atomic_pickle(obj, path: Path):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def retrieve_candidates():
    """Country-partitioned TF-IDF retrieval.
    For each country bucket, only query-rows from that country are multiplied
    against corpus chunks from that country. This gives ~10x fewer comparisons
    with ZERO recall loss (no cross-country matches exist in this dataset).
    """
    CAND_CACHE_V2 = CACHE / "tfidf_candidates_v2.pkl"

    # ── Load country-chunk meta from Cell 5 ──────────────────────────────────
    import json as _json
    try:
        with open(CHUNK_META) as f:
            _meta = _json.load(f)
    except FileNotFoundError:
        _meta = {}
    country_chunk_meta = _meta.get("country_chunk_meta", {})
    if not country_chunk_meta:
        # Fallback: treat all chunks as one country bucket (old behaviour)
        print("  [WARN] No country_chunk_meta found — falling back to global retrieval")
        country_chunk_meta = {"_ALL_": {
            "chunk_files": [str(p) for p in corpus_chunk_files],
            "corpus_ids":  corpus_ids.tolist(),
        }}

    # ── Build s1 country index ────────────────────────────────────────────────
    import pandas as _pd
    _s1_df = _pd.read_parquet(INPUT["source1"], columns=["entity_id", "country"]) if str(INPUT["source1"]).endswith(".parquet") else _pd.read_csv(INPUT["source1"], sep="\t", usecols=["entity_id", "country"])
    if DEBUG_MODE:
        _s1_df = _s1_df.iloc[:DEBUG_SAMPLE]
    _qid2country = dict(zip(_s1_df["entity_id"], _s1_df["country"]))
    del _s1_df; gc.collect()
    print(f"  Query country map: {len(_qid2country):,} entries")

    # Build per-country query row indices into s1_matrix
    from collections import defaultdict as _dd
    country_q_indices = _dd(list)
    for qi, qid in enumerate(s1_ids):
        ctry = _qid2country.get(qid, "UNKNOWN")
        country_q_indices[ctry].append(qi)
    print(f"  Query countries: { {k: len(v) for k, v in country_q_indices.items()} }")

    # ── Resume from checkpoint ─────────────────────────────────────────────────
    best_per_query = [dict() for _ in range(s1_matrix.shape[0])]
    global_chunk_done = set()

    if CAND_CACHE_V2.exists() and not DEBUG_MODE:
        try:
            with open(CAND_CACHE_V2, "rb") as f:
                ckpt = pickle.load(f)
            global_chunk_done = set(ckpt.get("chunks_done", []))
            _id2cidx = {eid: i for i, eid in enumerate(corpus_ids)}
            _id2qidx = {qid: i for i, qid in enumerate(s1_ids)}
            for qid, cid_score_pairs in ckpt["candidate_lists"].items():
                qi = _id2qidx.get(qid)
                if qi is None: continue
                for item in cid_score_pairs:
                    if isinstance(item, (list, tuple)) and len(item) == 2:
                        cid, real_sc = item[0], float(item[1])
                    else:
                        cid, real_sc = item, 0.5
                    ci = _id2cidx.get(cid)
                    if ci is not None:
                        best_per_query[qi][ci] = real_sc
            del _id2cidx, _id2qidx; gc.collect()
            print(f"  Resumed: {len(global_chunk_done)} chunks already done")
        except Exception as e:
            print(f"  Checkpoint unreadable ({e}) — starting fresh")
            best_per_query    = [dict() for _ in range(s1_matrix.shape[0])]
            global_chunk_done = set()

    # ── Build global corpus ID index (cid -> global corpus row index) ─────────
    _cid2cidx = {cid: i for i, cid in enumerate(corpus_ids)}

    t_total   = time.time()
    ckpt_ctr  = 0
    import zipfile, io as _io

    def _npz_nrows(path):
        with zipfile.ZipFile(path, "r") as zf:
            with zf.open("shape.npy") as f:
                return int(np.load(_io.BytesIO(f.read()))[0])

    countries = sorted(country_chunk_meta.keys())
    total_chunks = sum(len(v["chunk_files"]) for v in country_chunk_meta.values())
    print(f"  {total_chunks} total chunks across {len(countries)} countries")

    global_ci = 0
    for ctry in countries:
        ctry_meta   = country_chunk_meta[ctry]
        ctry_cfiles = [Path(p) for p in ctry_meta["chunk_files"]]
        ctry_cids   = np.array(ctry_meta["corpus_ids"])
        q_indices   = np.array(country_q_indices.get(ctry, []), dtype=np.int64)

        if len(q_indices) == 0:
            print(f"  [{ctry}] No queries — skipping")
            global_ci += len(ctry_cfiles)
            continue

        # Sub-matrix: only this country's query rows
        s1_ctry = s1_matrix[q_indices]   # shape: (n_ctry_queries, vocab)
        n_q_ctry = s1_ctry.shape[0]

        # Pre-compute per-chunk corpus offsets for this country
        ctry_offsets = []
        c_off = 0
        for cf in ctry_cfiles:
            nrows = _npz_nrows(cf)
            ctry_offsets.append(c_off)
            c_off += nrows

        print(f"  [{ctry}] {n_q_ctry:,} queries x {len(ctry_cfiles)} chunks ({len(ctry_cids):,} corpus docs)")

        for local_ci, (cf, c_start) in enumerate(zip(ctry_cfiles, ctry_offsets)):
            chunk_key = str(cf)
            if chunk_key in global_chunk_done:
                print(f"    [SKIP] {cf.name}")
                global_ci += 1
                continue

            t0 = time.time()
            print(f"    [{ctry}] Chunk {local_ci+1}/{len(ctry_cfiles)} Loading...", flush=True)
            # Fast read over network mount to avoid zipfile seek latency
            import io
            with open(cf, "rb") as _f:
                corpus_chunk = sp.load_npz(io.BytesIO(_f.read())).tocsr()

            if _USE_TOPN:
                sims = sp_matmul_topn(
                    s1_ctry,
                    corpus_chunk.T,
                    top_n=TOP_K_CANDIDATES,
                    threshold=float(TFIDF_THRESHOLD),
                    n_threads=4,
                )
            else:
                BATCH = 50_000
                parts = []
                for bs in range(0, n_q_ctry, BATCH):
                    part = s1_ctry[bs:min(bs+BATCH, n_q_ctry)].dot(corpus_chunk.T).tocsr()
                    part.data[part.data < TFIDF_THRESHOLD] = 0
                    part.eliminate_zeros()
                    parts.append(part)
                sims = sp.vstack(parts, format="csr")
                del parts; gc.collect()

            iptr = sims.indptr
            idxs = sims.indices
            dta  = sims.data

            for local_qi in range(n_q_ctry):
                rs, re = iptr[local_qi], iptr[local_qi + 1]
                if rs == re: continue
                sc_arr  = dta[rs:re]
                # local_ci's corpus chunk -> map local col idx -> global corpus idx
                lc_arr  = idxs[rs:re]
                gc_arr  = np.array([_cid2cidx[ctry_cids[lc]] for lc in lc_arr.tolist()], dtype=np.int64)
                best    = best_per_query[q_indices[local_qi]]

                if len(sc_arr) <= TOP_K_CANDIDATES:
                    for g_idx, sc in zip(gc_arr.tolist(), sc_arr.tolist()):
                        if g_idx not in best or sc > best[g_idx]:
                            best[g_idx] = sc
                else:
                    tk = np.argpartition(sc_arr, -TOP_K_CANDIDATES)[-TOP_K_CANDIDATES:]
                    for idx in tk.tolist():
                        g_idx = int(gc_arr[idx])
                        sc = float(sc_arr[idx])
                        if g_idx not in best or sc > best[g_idx]:
                            best[g_idx] = sc

            load_time = time.time() - t0
            del corpus_chunk, sims; gc.collect()
            global_chunk_done.add(chunk_key)
            global_ci += 1
            print(f"    [{ctry}] Chunk {local_ci+1}/{len(ctry_cfiles)} done in {load_time:.1f}s "
                  f"({global_ci}/{total_chunks} total)", flush=True)
            mem_report(f"After {ctry} chunk {local_ci+1}")

            # ── Rolling checkpoint every 5 chunks ────────────────────────────
            ckpt_ctr += 1
            if not DEBUG_MODE and ckpt_ctr % 5 == 0:
                print(f"    [CKPT] Saving ...")
                _partial = {}
                for qi, best in enumerate(best_per_query):
                    if not best: continue
                    cols   = np.array(list(best.keys()), dtype=np.int64)
                    scores = np.array(list(best.values()), dtype=np.float32)
                    if len(scores) > TOP_K_CANDIDATES:
                        tk     = np.argpartition(scores, -TOP_K_CANDIDATES)[-TOP_K_CANDIDATES:]
                        cols   = cols[tk]; scores = scores[tk]
                    _partial[s1_ids[qi]] = list(zip(corpus_ids[cols].tolist(), scores.tolist()))
                atomic_pickle({"candidate_lists": _partial,
                               "chunks_done": list(global_chunk_done)}, CAND_CACHE_V2)
                del _partial; gc.collect()
                print(f"    [CKPT] Done.")

        del s1_ctry; gc.collect()

    del _cid2cidx; gc.collect()

    # ── Finalise top-K ────────────────────────────────────────────────────────
    print("\nFinalising top-K candidate lists ...")
    candidate_lists = {}
    all_cands = 0
    for qi, best in enumerate(best_per_query):
        if not best:
            candidate_lists[s1_ids[qi]] = []
            continue
        cols   = np.array(list(best.keys()), dtype=np.int64)
        scores = np.array(list(best.values()), dtype=np.float32)
        if len(scores) > TOP_K_CANDIDATES:
            tk    = np.argpartition(scores, -TOP_K_CANDIDATES)[-TOP_K_CANDIDATES:]
            tk    = tk[np.argsort(-scores[tk])]
            cols  = cols[tk]; scores = scores[tk]
        else:
            order = np.argsort(-scores)
            cols  = cols[order]; scores = scores[order]
        candidate_lists[s1_ids[qi]] = corpus_ids[cols].tolist()
        all_cands += len(cols)

    del best_per_query; gc.collect()
    elapsed = (time.time() - t_total) / 60
    print(f"\nRetrieval done in {elapsed:.1f} min")
    print(f"  {len(candidate_lists):,} queries | {all_cands:,} candidates "
          f"({all_cands / max(len(candidate_lists), 1):.1f} avg/query)")

    if not DEBUG_MODE:
        _final_ckpt = {}
        for qid, cids in candidate_lists.items():
            _final_ckpt[qid] = [(cid, 0.5) for cid in cids]
        atomic_pickle({"candidate_lists": _final_ckpt,
                       "chunks_done": list(global_chunk_done)}, CAND_CACHE_V2)
        del _final_ckpt; gc.collect()
    return candidate_lists

candidate_lists = retrieve_candidates()
del s1_matrix; gc.collect()
mem_report("After Retrieval")


In [ ]:
# ==============================================================================
# Cell 7: Stage 2 — Cross-Encoder with Full Raw Score Caching
#
# KEY DESIGN CHANGE vs old code:
#   OLD: Apply threshold immediately, throw away raw scores.
#        → To change CE_THRESHOLD you MUST re-run 4-5 hours of GPU inference.
#   NEW: Save ALL raw (cid, score) pairs per block to disk first.
#        → Threshold tuning = re-run apply_threshold() in ~5 minutes, NOT re-inference.
#
# Cache structure:  CACHE/ce_scores_raw/block_XXXXX.pkl
#   Each file: dict {qid: [(cid, float_score), ...]}  — full list, no threshold applied.
#
# Resume: Block completion = file existence check (NOT a 1.73M-entry JSON).
#   Old progress.json grew to 150MB JSON write per block — removed entirely.
#
# OOM risks addressed:
#   FM4: pairs built per-block only (25k queries × 15 cands = 375k pairs max)
#   FM5: CUDA OOM exponential backoff (256→128→64→32→16)
#   FM9: corpus_text_lookup (~2GB RAM) freed after inference loop ends
# ==============================================================================

CE_SCORES_DIR = CACHE / "ce_scores_raw"
CE_SCORES_DIR.mkdir(exist_ok=True)

def _block_path(bi):
    return CE_SCORES_DIR / f"block_{bi:05d}.pkl"

def run_cross_encoder():
    """Run CE inference. Saves raw scores per block. Resumes from last saved block."""
    global corpus_text_lookup, s1_text_lookup

    n_blocks  = (len(s1_ids) + QUERY_BLOCK - 1) // QUERY_BLOCK
    remaining = list(s1_ids)   # all queries in order

    # Resume: which blocks already have score files on disk?
    done_set  = {bi for bi in range(n_blocks) if _block_path(bi).exists()}
    todo      = [bi for bi in range(n_blocks) if bi not in done_set]

    print(f"CE blocks total: {n_blocks}  |  cached: {len(done_set)}  |  todo: {len(todo)}")

    if not todo:
        print("All blocks already cached! Skipping inference — going straight to threshold.")
        del corpus_text_lookup, s1_text_lookup; gc.collect()
        print("  Freed text lookup dicts.")
        return

    # ── Load model ─────────────────────────────────────────────────────────────
    print(f"\nLoading {CE_MODEL_NAME} ...")
    from sentence_transformers import CrossEncoder
    device    = "cuda" if torch.cuda.is_available() else "cpu"
    model     = CrossEncoder(CE_MODEL_NAME, max_length=CE_MAX_LEN, device=device)
    if device == "cuda":
        try:
            model.model = model.model.half()
            print("  FP16 enabled.")
        except Exception as e:
            print(f"  FP16 note: {e}")
    print(f"  Device: {device}  |  initial batch size: {CE_BATCH_INIT}")

    current_bs  = CE_BATCH_INIT
    total_start = time.time()

    for bi in todo:
        bstart     = bi * QUERY_BLOCK
        bend       = min((bi + 1) * QUERY_BLOCK, len(remaining))
        block_qids = remaining[bstart:bend]

        print(f"\n--- Block {bi+1}/{n_blocks} ({len(block_qids):,} queries) "
              f"batch_size={current_bs} ---", flush=True)

        # FM4: build pairs ONLY for this block — never globally
        block_pairs    = []
        block_pair_map = []         # (qid, cid) tuples
        for qid in block_qids:
            qt = s1_text_lookup[qid]
            for cid in candidate_lists.get(qid, []):
                block_pairs.append([qt, corpus_text_lookup[cid]])
                block_pair_map.append((qid, cid))

        # raw scores dict: {qid: [(cid, score), ...]} — threshold NOT applied yet
        block_raw = {qid: [] for qid in block_qids}

        if block_pairs:
            print(f"  Scoring {len(block_pairs):,} pairs ...", flush=True)
            t0      = time.time()
            success = False
            scores  = None

            while not success:
                try:
                    with torch.no_grad():
                        scores = model.predict(
                            block_pairs,
                            batch_size=current_bs,
                            show_progress_bar=True,
                            convert_to_numpy=True)
                    if scores.ndim > 1:
                        scores = scores.flatten()
                    success = True
                except torch.cuda.OutOfMemoryError:
                    print(f"  [CUDA OOM] batch_size={current_bs} → halving ...")
                    torch.cuda.empty_cache(); gc.collect()
                    current_bs = max(16, current_bs // 2)
                    print(f"  Retrying with batch_size={current_bs} ...", flush=True)

            elapsed = time.time() - t0
            print(f"  Scored in {elapsed:.1f}s  ({len(block_pairs)/elapsed:.0f} pairs/s)",
                  flush=True)

            # Store ALL raw scores — NO threshold, NO filtering yet
            for (qid, cid), sc in zip(block_pair_map, scores.tolist()):
                block_raw[qid].append((cid, float(sc)))

        # Atomic save — if run is killed mid-write, corrupt .tmp never replaces .pkl
        _tmp = _block_path(bi).with_suffix(".tmp")
        with open(_tmp, "wb") as f:
            pickle.dump(block_raw, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(_tmp, _block_path(bi))
        print(f"  [SAVED] {_block_path(bi).name}", flush=True)

        del block_pairs, block_pair_map, block_raw
        if "scores" in locals(): del scores
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()
        mem_report(f"End block {bi+1}/{n_blocks}")

    total_time = (time.time() - total_start) / 60
    print(f"\nAll CE inference complete in {total_time:.1f} min!")

    # FM9: free the ~2GB text lookups — not needed by apply_threshold()
    del corpus_text_lookup, s1_text_lookup; gc.collect()
    print("Freed text lookup dicts (~2 GB RAM recovered).")


def apply_threshold(threshold=CE_THRESHOLD):
    """
    FAST threshold application on cached CE scores.

    Reads block_{XXXXX}.pkl files from disk, applies threshold, writes TSV.
    Runs in ~5 minutes — no GPU, no model load, no re-inference.

    Usage for threshold tuning (run this cell again with a new value):
        apply_threshold(0.75)   # lower  → more matches, higher recall
        apply_threshold(0.85)   # higher → fewer matches, higher precision
        apply_threshold(0.90)

    Each call overwrites PARTIAL_TSV with the new thresholded output.
    Then re-run Cell 8 to produce a fresh submission ZIP.
    """
    n_blocks    = (len(s1_ids) + QUERY_BLOCK - 1) // QUERY_BLOCK
    block_files = [_block_path(bi) for bi in range(n_blocks)]
    missing     = [bf.name for bf in block_files if not bf.exists()]

    if missing:
        raise RuntimeError(
            f"{len(missing)} block score files are missing — run run_cross_encoder() first.\n"
            f"  First missing: {missing[0]}")

    print(f"Applying CE_THRESHOLD={threshold} to {n_blocks} cached blocks ...")
    print(f"  Blocks directory: {CE_SCORES_DIR}")

    # Overwrite TSV fresh for this threshold
    with open(PARTIAL_TSV, "w", encoding="utf-8") as f:
        f.write("source1_entity_id\tmatched_entity_ids\n")

    total_matched = 0
    total_queries = 0
    t0 = time.time()

    for bi, bf in enumerate(block_files):
        with open(bf, "rb") as f:
            block_raw = pickle.load(f)

        with open(PARTIAL_TSV, "a", encoding="utf-8") as f:
            for qid, pairs in block_raw.items():
                # Apply threshold + validity filter
                matched = [
                    cid for cid, sc in pairs
                    if sc >= threshold and cid.startswith(("S2-", "S3-"))
                ]
                # Deduplicate (preserve order)
                seen_m, unique = set(), []
                for m in matched:
                    if m not in seen_m:
                        seen_m.add(m); unique.append(m)
                f.write(f"{qid}\t{','.join(unique)}\n")
                total_matched += len(unique)
                total_queries += 1

        del block_raw; gc.collect()

        if (bi + 1) % 10 == 0 or bi == n_blocks - 1:
            pct = (bi + 1) / n_blocks * 100
            print(f"  [{pct:5.1f}%] {bi+1}/{n_blocks} blocks processed ...", flush=True)

    elapsed = (time.time() - t0) / 60
    print(f"\nThreshold application done in {elapsed:.1f} min")
    print(f"  {total_queries:,} queries | {total_matched:,} total matches "
          f"({total_matched / max(total_queries, 1):.2f} avg/query)")
    print(f"  TSV: {PARTIAL_TSV}")


# ── Run ────────────────────────────────────────────────────────────────────────
run_cross_encoder()          # inference (resumes if partially done)
apply_threshold(CE_THRESHOLD)   # apply threshold from Cell 2 config
# To retune: just change CE_THRESHOLD in Cell 2 and re-run apply_threshold()



In [ ]:
# ==============================================================================
# Cell 8: Finalize Submission Files
# ==============================================================================
import shutil

tsv_out  = OUT  / "matching_results.tsv"
tsv_work = WORK / "matching_results.tsv"
zip_out  = WORK / "submission.zip"

shutil.copyfile(PARTIAL_TSV, tsv_out)
shutil.copyfile(PARTIAL_TSV, tsv_work)

mb = tsv_out.stat().st_size / 1e6
print(f"Final TSV: {tsv_out}  ({mb:.1f} MB)")

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(tsv_out, arcname="matching_results.tsv")
print(f"ZIP:       {zip_out}  ({zip_out.stat().st_size/1e6:.1f} MB)")


In [ ]:
# ==============================================================================
# Cell 9: Built-in Official Submission Validator
# Re-define paths in case cell is run independently
tsv_out  = OUT / "matching_results.tsv"
# ==============================================================================
def validate(tsv_path: Path, expected_ids: set):
    errors, n_rows, empty = [], 0, 0
    seen, dup, intra, self_m, bad_pfx = set(), set(), set(), set(), set()

    with open(tsv_path, encoding="utf-8") as f:
        hdr = f.readline()
        if not hdr:
            errors.append("File is empty"); return errors
        if "\t" not in hdr and "," in hdr:
            errors.append("File looks CSV, not TSV"); return errors
        cols = [c.strip().lower() for c in hdr.rstrip("\n").split("\t")]
        if cols != ["source1_entity_id","matched_entity_ids"]:
            errors.append(f"Bad header: {cols}"); return errors

        for line in f:
            parts = line.rstrip("\r\n").split("\t")
            if len(parts) != 2: errors.append(f"Malformed row: {line[:80]!r}"); continue
            s1, mstr = parts
            n_rows += 1
            if s1 in seen: dup.add(s1)
            seen.add(s1)
            if not mstr.strip(): empty += 1; continue
            ml = mstr.split(",")
            if len(ml) != len(set(ml)): intra.add(s1)
            for m in ml:
                if   m.startswith("S1-"):          self_m.add(m)
                elif not m.startswith(("S2-","S3-")): bad_pfx.add(m)

    if dup:     errors.append(f"Duplicate S1 rows: {len(dup)}")
    if intra:   errors.append(f"Intra-list duplicate IDs: {len(intra)} rows")
    if self_m:  errors.append(f"Self-matches (S1-) found: {len(self_m)}")
    if bad_pfx: errors.append(f"Invalid ID prefix: {len(bad_pfx)}")
    miss = expected_ids - seen
    if miss:    errors.append(f"Missing S1 entities: {len(miss)}")
    extra = seen - expected_ids
    if extra:   errors.append(f"Unexpected S1 rows: {len(extra)}")

    print("=" * 55)
    print("VALIDATION REPORT")
    print(f"  Total rows      : {n_rows:,}")
    print(f"  With matches    : {n_rows-empty:,}")
    print(f"  Without matches : {empty:,}")
    print(f"  Errors found    : {len(errors)}")
    if errors:
        print("\n[FAILED]")
        for e in errors: print(f"  - {e}")
    else:
        print("\n[PASSED] 100% competition compliant!")
    print("=" * 55)
    return errors

errs = validate(tsv_out, set(s1_ids))


In [ ]:
# ==============================================================================
# Cell 10: Download Links
# ==============================================================================
from IPython.display import FileLink, display, HTML
display(FileLink(str(tsv_work),  result_html_prefix="TSV: "))
display(FileLink(str(zip_out),   result_html_prefix="ZIP: "))

display(HTML("""
<div style="background:#0f172a;color:#f8fafc;padding:14px;border-radius:8px;
            border:1px solid #38bdf8;margin-top:12px">
<b style="color:#38bdf8">Kaggle sidebar:</b>
Data → Output → /kaggle/working → matching_results.tsv → ⋮ → Download
</div>"""))
